In [21]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from utils import multi_hot_encode, prepare_recommender_data, load_loss_val, create_prediction_results
from Models.recommender_model_code import MovieRecommender

RANDOM_STATE = 36
SEED = 1234

# 1. Importing Data

In [22]:
movies = pd.read_csv('./MovieLens/movies.csv')
ratings = pd.read_csv('./MovieLens/ratings.csv')

print(movies.shape, ratings.shape)
print(movies.columns, ratings.columns)

(9742, 3) (100836, 4)
Index(['movieId', 'title', 'genres'], dtype='str') Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')


In [23]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [24]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


# 2. Encoding Genres Column

In [25]:
genres = [
    "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir",
    "Horror", "IMAX", "Musical", "Mystery", "Romance",
    "Sci-Fi", "Thriller", "War", "Western"
]
genre_col = movies['genres']

encoded_columns = multi_hot_encode(genres, genre_col)

movies['genres_encoded'] = encoded_columns


# Mergrin encoded Gernes to Ratings DataFrame

In [26]:
ratings = ratings.merge(
    movies[['movieId', 'genres_encoded']], # Columns
    how='left', # Ratings is the main Df
    on='movieId' # Match based on movieId column
)

# Dropping TimeStamp Column

In [27]:
ratings = ratings.drop('timestamp', axis=1)

In [28]:
ratings

,userId,movieId,rating,genres_encoded
0,1,1,4.0,"[0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
1,1,3,4.0,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
2,1,6,4.0,"[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,1,47,5.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
4,1,50,5.0,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
...,...,...,...,...
100831,610,166534,4.0,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, ..."
100832,610,168248,5.0,"[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
100833,610,168250,5.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
100834,610,168252,5.0,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# Train/Validation Splitting Data

In [29]:
train_data , cv_data = train_test_split(ratings, test_size=0.2, random_state=RANDOM_STATE)

print(train_data.shape, cv_data.shape)
cv_data.head()

(80668, 4) (20168, 4)


,userId,movieId,rating,genres_encoded
43714,292,4643,3.0,"[1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
40452,274,59604,3.5,"[0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, ..."
93719,599,3735,3.0,"[0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
63442,414,3535,4.0,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, ..."
60720,391,3717,2.0,"[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# Preparing Recommender NN Data

In [30]:
X_user_train, y_train, X_movie_train, train_movie_ids, X_user_cv, X_movie_cv, y_cv, cv_movie_ids = prepare_recommender_data(train_data, cv_data)

X_user_train.shape, y_train.shape, X_movie_train.shape, train_movie_ids.shape, X_user_cv.shape, X_movie_cv.shape, y_cv.shape, cv_movie_ids.shape

((80668,),
 (80668,),
 (80668, 19),
 (80668,),
 (20168,),
 (20168, 19),
 (20168,),
 (20168,))

---
# Training Model

In [31]:
num_users = np.unique(X_user_train).shape[0] # 610

model = tf.keras.models.load_model('Models/pro4_changed_architecture&l2reg.keras')
# model = MovieRecommender(num_users=num_users, l2_lambda=0)

model.summary()

Model: "movie_recommender"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ user_model (Sequential)         │ (None, 32)             │        19,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ movie_model (Sequential)        │ (None, 32)             │        42,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ interaction_model (Sequential)  │ (None, 1)              │         6,273 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,909 (796.52 KB)

 Trainable params: 67,969 (265.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 135,940 (531.02 KB)

## Compile & Fit
Saved The model Since it took me 5 Minutes each time to train it

In [32]:
# model.compile(
#     optimizer = tf.keras.optimizers.Adam(0.001),
#     loss = tf.keras.losses.MeanSquaredError()
# )

# early_stopping = tf.keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=3,
#     restore_best_weights=True
# )

# history = model.fit(
#     [X_user_train.reshape(-1, 1), X_movie_train], y_train,
#     validation_data=([X_user_cv.reshape(-1, 1), X_movie_cv], y_cv),
#     epochs=20,
#     callbacks=[early_stopping]
# )


# Making Predictions

In [34]:
# pred_train = model.predict([X_user_train, X_movie_train])
# pred_cv = model.predict([X_user_cv, X_movie_cv])

predictions = np.load('Model Predictions/predictions.npz')

pred_train = predictions['pred_train']
pred_cv = predictions['pred_cv']

In [ ]:
# np.savez(
#     'predictions.npz',
#     pred_train = pred_train,
#     pred_cv=pred_cv
# )

In [36]:
mse_train = mean_squared_error(y_train, pred_train)
mse_cv = mean_squared_error(y_cv, pred_cv)
mse_train, mse_cv

(0.7462937695813061, 0.8026744830322161)

# Comparing Prediciton vs Actual Ratings

In [55]:
comparison_train = create_prediction_results(train_data, movies, pred_train)

comparison_train.head()

,userId,movieId,rating,predicted_rating,title,genres
0,438,3994,4.5,3.3,Unbreakable (2000),Drama|Sci-Fi
1,73,111921,3.5,3.8,The Fault in Our Stars (2014),Drama|Romance
2,15,158,1.0,2.7,Casper (1995),Adventure|Children
3,486,153,5.0,3.7,Batman Forever (1995),Action|Adventure|Comedy|Crime
4,29,914,4.0,4.0,My Fair Lady (1964),Comedy|Drama|Musical|Romance


In [56]:
print(f'UserId | Movie | Actual Rating | Predicted Rating')
for i, rows in comparison_train[:5].iterrows():
    print(
        f'{ rows['userId']} | {rows['title']} | {rows['rating']} | {np.round(rows['predicted_rating'], 1)}'
    )

UserId | Movie | Actual Rating | Predicted Rating
438 | Unbreakable (2000) | 4.5 | 3.3
73 | The Fault in Our Stars (2014) | 3.5 | 3.8
15 | Casper (1995) | 1.0 | 2.7
486 | Batman Forever (1995) | 5.0 | 3.7
29 | My Fair Lady (1964) | 4.0 | 4.0
